# Finite quotient algebras

This notebook uses the Python binding and its optional structured SymPy adapter. Install `sympy` beside the `sylvester` package before running it.

In [ ]:
import sympy as sp
import sylvester
from sylvester.sympy import from_sympy, to_sympy

## A verified quotient over a prime field

The certificate path proves the input and returned bases generate the same ideal over `F_p`. The quotient below has four standard monomials.

In [ ]:
p = 101
ring = sylvester.PolynomialRing.prime_field(p, ["x", "y"])
x_sym, y_sym = sp.symbols("x y")


def prime(expr):
    return from_sympy(sp.Poly(expr, x_sym, y_sym, modulus=p), ring)


x = prime(x_sym)
y = prime(y_sym)
ideal = ring.ideal([prime(x_sym**2), prime(y_sym**2)])
assert to_sympy(x) == sp.Poly(x_sym, x_sym, y_sym, modulus=p)

In [ ]:
certified = ideal.groebner_basis_certified()
basis = certified.basis
verified = sylvester.verify(certified.certificate)
assert verified.modulus == p

quotient = basis.finite_quotient()
assert quotient.dimension == 4
assert {tuple(exponents) for exponents in quotient.standard_monomials} == {
    (0, 0),
    (1, 0),
    (0, 1),
    (1, 1),
}

The relations `x^2 = 0` and `y^2 = 0` reduce every residue to the span of `1`, `x`, `y`, and `x*y`.

In [ ]:
residue = quotient.reduce(prime(x_sym**3 + x_sym*y_sym + 1))
assert residue == prime(x_sym*y_sym + 1)
coordinates = quotient.coordinates(residue)
assert len(coordinates) == quotient.dimension

matrix = quotient.multiplication_matrix(x)
assert matrix.dimension == 4
matrix.entries

## Nilpotents separate the two spectra

Multiplication by `x` is nilpotent. Its minimal polynomial records the first relation, `t^2`, while its characteristic polynomial records the full four-dimensional operator, `t^4`.

In [ ]:
characteristic = quotient.characteristic_polynomial(x)
minimal = quotient.minimal_polynomial(x)
assert str(characteristic) == "t^4"
assert str(minimal) == "t^2"
characteristic, minimal

## Rational computations

The rational driver is multimodular and heuristic. `contains_input` records that the returned basis defines an ideal containing the input ideal. It does not establish ideal equality.

In [ ]:
qring = sylvester.PolynomialRing.rationals(["x", "y"])
qx_sym, qy_sym = sp.symbols("x y")


def rational(expr):
    return from_sympy(sp.Poly(expr, qx_sym, qy_sym, domain=sp.QQ), qring)


qideal = qring.ideal(
    [
        rational(qx_sym**2 + qy_sym**2 - 1),
        rational(4*qx_sym*qy_sym - 1),
    ]
)
qbasis = qideal.groebner_basis(stop="contains_input")
qquotient = qbasis.finite_quotient()
qminimal = qquotient.minimal_polynomial(rational(qx_sym))

assert str(qminimal) == "t^4 - t^2 + 1/16"
assert qbasis.lift()["established"] == "contains_input"

The SymPy adapter accepts `Poly` over `QQ`, `ZZ`, and prime `GF(p)`. It keeps the generator order and exact coefficients. It rejects inexact, extension, and composite finite-field domains.